# Extreme Solar Events — Implementation / 구현

**Paper**: Cliver, E. W., Schrijver, C. J., Shibata, K., & Usoskin, I. G. (2022). Extreme solar events. *Living Reviews in Solar Physics*, 19(1), 2. DOI: 10.1007/s41116-022-00033-8

This notebook reproduces the key quantitative results of the Cliver et al. (2022) review of extreme solar events. It covers: (1) flare energy power-law distribution with rolloff, (2) Gumbel extreme value analysis of a synthetic flare record, (3) return period for Carrington-class events, (4) Miyake Δ¹⁴C time series including the 774, 993, and 660 BCE events, (5) Kepler superflare frequency on Sun-like stars, and (6) infrastructure-loss estimation as a function of event magnitude.

이 노트북은 Cliver et al. (2022)의 극한 태양 사건 리뷰의 핵심 정량적 결과를 재현한다. (1) rolloff 포함 플레어 에너지 멱법칙 분포, (2) 합성 플레어 기록에 대한 Gumbel 극값 분석, (3) Carrington급 사건의 return period, (4) 774, 993, 660 BCE 사건을 포함하는 Miyake Δ¹⁴C 시계열, (5) Sun-like 별의 Kepler 슈퍼플레어 빈도, (6) 사건 규모에 따른 인프라 손실 추정을 다룬다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
rng = np.random.default_rng(42)

## Part 1: Flare Energy Power-Law with Rolloff / 플레어 에너지 멱법칙 + rolloff

The solar flare energy distribution follows $dN/dE \sim E^{-\alpha}$ with $\alpha \approx 1.8$ across ~14 orders of magnitude (Aschwanden et al. 2016). For extreme events, a rolloff at high energies appears necessary to constrain the 100-year and 1000-year event sizes. Cliver et al. adopt a **modified exponential** form from Gopalswamy (2018) which fits better than a pure power-law over the full range of observations.

태양 플레어 에너지 분포는 $dN/dE \sim E^{-\alpha}$ ($\alpha \approx 1.8$)를 ~14자릿수에 걸쳐 따른다 (Aschwanden 2016). 극한 사건의 경우 고에너지에서의 rolloff가 100년/1000년 사건 크기를 제약하는 데 필요해 보인다. Cliver et al.은 관측 전 범위에 걸쳐 순수 멱법칙보다 더 잘 맞는 Gopalswamy (2018)의 수정 지수 형태를 채택한다.

In [ ]:
def power_law_pdf(E, alpha, E_min):
    """Power-law probability density dN/dE ~ E^(-alpha).

    Args:
        E: Energy values (erg).
        alpha: Power-law index.
        E_min: Lower cutoff.

    Returns:
        Normalized PDF values (arb. units).
    """
    return (alpha - 1) * E_min**(alpha - 1) * E**(-alpha)


def power_law_with_rolloff(E, alpha, E_cutoff, E_min):
    """Power-law with exponential rolloff.

    Args:
        E: Energy values (erg).
        alpha: Power-law index below the rolloff.
        E_cutoff: Exponential cutoff energy.
        E_min: Lower cutoff.

    Returns:
        PDF values incorporating the rolloff (arb. units).
    """
    return (E / E_min)**(-alpha) * np.exp(-E / E_cutoff)


def modified_exponential_cumulative(X, a, b, c):
    """Gopalswamy (2018) modified exponential OFD (Eq. 1 of paper).

    Y = a * (1 - exp(-(-X + b)/c)), downward cumulative count in log units.

    Args:
        X: log10(event size).
        a: Normalization.
        b: Location.
        c: Scale.

    Returns:
        Y = log10(cumulative count per year).
    """
    return a * (1.0 - np.exp(-(-X + b) / c))


# Build the flare energy distribution
E = np.logspace(28, 36, 500)
alpha = 1.8  # Aschwanden universal slope
E_min = 1e28
E_cutoff_10kyr = 2e33   # rolloff near 10,000-yr event (Fig. 8 estimate)

pdf_pure = power_law_pdf(E, alpha, E_min)
pdf_roll = power_law_with_rolloff(E, alpha, E_cutoff_10kyr, E_min)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(E, pdf_pure, 'r--', label=r'Pure power-law  $dN/dE \propto E^{-1.8}$')
ax.loglog(E, pdf_roll, 'b-', label=r'With exponential rolloff at $E_c = 2 \times 10^{33}$ erg')
ax.axvline(4.3e32, color='k', ls=':', lw=1.2, label='4 Nov 2003 (X35, $4.3\\times 10^{32}$ erg)')
ax.axvline(5e32, color='g', ls=':', lw=1.2, label='Carrington 1859 (X45, $\\sim 5\\times 10^{32}$ erg)')
ax.axvline(1.9e33, color='m', ls=':', lw=1.2, label='774 AD inferred (X285, $1.9\\times 10^{33}$ erg)')
ax.axvline(1e34, color='orange', ls=':', lw=1.2, label='Kepler superflare threshold ($10^{34}$ erg)')
ax.set_xlabel('Flare bolometric energy $E$ (erg) / 플레어 복사 에너지')
ax.set_ylabel(r'$dN/dE$ (arb. units)')
ax.set_title('Solar flare energy distribution with rolloff / 플레어 에너지 분포와 rolloff')
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(1e-45, 1e-27)
plt.tight_layout()
plt.show()

## Part 2: Gumbel Extreme Value Fit / Gumbel 극값 분포 적합

Extreme value theory (GEV distribution) classifies extreme tails into Gumbel (light tail, $\xi = 0$), Fréchet (heavy tail, $\xi > 0$), and Weibull-type (bounded, $\xi < 0$). For annual block maxima of solar flare peak flux, Tsubouchi & Omura (2007) and Thomson et al. (2011) used extreme value theory to estimate 100-year storm amplitudes.

Here we simulate a synthetic record of annual largest flare peak fluxes (in log space) drawn from a truncated power-law, and fit a Gumbel distribution.

극값 이론(GEV 분포)은 극한 꼬리를 Gumbel (light tail), Fréchet (heavy tail), Weibull-type (bounded)로 분류한다. 태양 플레어 피크 플럭스의 연간 블록 최대값에 대해 Tsubouchi & Omura (2007), Thomson et al. (2011)이 극값 이론으로 100년 폭풍 진폭을 추정했다. 여기서는 truncated 멱법칙에서 추출된 연간 최대 플레어 peak flux 합성 기록(로그 공간)에 Gumbel 분포를 적합한다.

In [ ]:
def draw_annual_max_flare(n_years=50, alpha=2.1, events_per_year=120, rng_state=None):
    """Simulate annual maximum log flare peak flux assuming PL with rolloff.

    Draws from a truncated Pareto and applies an exponential soft cutoff.

    Args:
        n_years: Number of synthetic years.
        alpha: Power-law index of the flare peak flux distribution.
        events_per_year: Approx. M- and X-class flares per year at solar max.
        rng_state: Numpy Generator instance.

    Returns:
        ndarray of log10(peak flux / X1) for the annual maxima.
    """
    if rng_state is None:
        rng_state = np.random.default_rng()
    annual_max = np.zeros(n_years)
    for i in range(n_years):
        # Pareto draws in X1 units (>= 1)
        samples = (rng_state.pareto(alpha - 1, events_per_year) + 1.0)
        # Apply exponential rolloff at X200 (soft cutoff for 10,000-yr event)
        keep = rng_state.uniform(size=events_per_year) < np.exp(-samples / 200.0)
        if keep.any():
            annual_max[i] = np.log10(samples[keep].max())
        else:
            annual_max[i] = np.log10(samples.max())
    return annual_max


annual_max = draw_annual_max_flare(n_years=50, rng_state=rng)

# Gumbel fit: fit MLE parameters mu, beta
mu, beta = stats.gumbel_r.fit(annual_max)
print(f"Gumbel location mu   = {mu:.3f}  (log10 X-class)")
print(f"Gumbel scale    beta = {beta:.3f}")
print(f"Observed max over {len(annual_max)} yrs: X{10**annual_max.max():.0f}")

# Plot empirical vs. fitted CDF
x_grid = np.linspace(annual_max.min() - 0.2, annual_max.max() + 1.5, 300)
cdf_fit = stats.gumbel_r.cdf(x_grid, loc=mu, scale=beta)
annual_max_sorted = np.sort(annual_max)
cdf_emp = np.arange(1, len(annual_max_sorted) + 1) / (len(annual_max_sorted) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(x_grid, cdf_fit, 'b-', label=f'Gumbel fit ($\\mu$={mu:.2f}, $\\beta$={beta:.2f})')
axes[0].plot(annual_max_sorted, cdf_emp, 'ko', mfc='none', label='Empirical (50 yr synth)')
axes[0].set_xlabel(r'$\log_{10}$(peak flux / X1)')
axes[0].set_ylabel('Cumulative probability')
axes[0].set_title('Gumbel fit to annual max flare flux / 연간 최대 플레어 플럭스의 Gumbel 적합')
axes[0].legend()

# Return-period plot
T = np.logspace(0, 4, 200)  # return periods 1 to 10,000 years
p = 1.0 - 1.0 / T
x_T = stats.gumbel_r.ppf(p, loc=mu, scale=beta)
axes[1].semilogx(T, 10**x_T, 'b-', lw=2, label='Gumbel return level')
axes[1].axhline(45, color='g', ls='--', label='X45 (Carrington)')
axes[1].axhline(285, color='m', ls='--', label='X285 (774 AD)')
axes[1].axhline(35, color='k', ls=':', label='X35 (4 Nov 2003)')
axes[1].set_xlabel('Return period T (years)')
axes[1].set_ylabel('Expected max X-class')
axes[1].set_title('Return-level curve / Return-level 곡선')
axes[1].legend()
plt.tight_layout()
plt.show()

## Part 3: Return Period for Carrington-Class Events / Carrington급 사건의 Return Period

From Cliver et al. (2022) Table 10: 100-year SXR class is X44 (exponential) or X42 (power-law); Carrington is estimated at X45 ± 5. Thus Carrington is approximately a **100-year event** in SXR class — consistent with the waiting-time study by Riley (2012).

For the Dst index, Carrington's storm (Dst ~ −949 nT) is between the 100-year (−603 nT exponential) and 1000-year (−845 nT exponential or −1470 nT power-law) event sizes. The fact that Carrington, 1872, and 1921 storms all exceeded the exponential-based 1000-year value within ~160 years suggests the power-law tail may be more appropriate.

Cliver et al. (2022) Table 10에서: 100년 SXR 등급은 X44 (exponential) 또는 X42 (power-law); Carrington은 X45 ± 5로 추정. 따라서 Carrington은 SXR 등급에서 대략 **100년 사건** — Riley (2012)의 waiting-time 연구와 일치. Dst 지수의 경우 Carrington 폭풍(Dst ~ −949 nT)은 100년(−603 nT) 및 1000년(−845 nT 또는 −1470 nT) 사건 크기 사이. Carrington, 1872, 1921 폭풍이 모두 ~160년 내 exponential 기반 1000년 값을 초과한 사실은 power-law 꼬리가 더 적절할 수 있음을 시사.

In [ ]:
def return_period_from_ofd(X_value, a, b, c):
    """Return period from Gopalswamy (2018) modified exponential OFD.

    Args:
        X_value: log10 of event size.
        a, b, c: Fit parameters.

    Returns:
        Return period in years.
    """
    Y = modified_exponential_cumulative(X_value, a, b, c)
    # Y = log10(annual rate); rate = 10**Y; T = 1/rate
    # But Gopalswamy OFD returns Y on a scale where Y = -2 maps to 1/100
    return 10.0**(-Y)


# Gopalswamy (2018) fit parameters for solar flare SXR OFD (see Fig. 8 in Cliver 2022)
# Fit in cumulative form: Y = a * (1 - exp((X+b)/c)) with the sign convention from paper
# For flare OFD Fig. 8 (Weibull): a=4.7, gamma=-3.3, eta=2.6, m=1.0.
# Here, instead of re-coding Weibull, we use tabulated 100- and 1000-yr estimates
# to illustrate the return-period concept.

flare_tab = {
    'X20  (4 Nov 2003, X35 after recalib → X50)': (35, 1.0),    # ~annual-to-decadal
    'X35  (4 Nov 2003 original)':                  (35, 10.0),
    'X45  (Carrington 1859)':                      (45, 100.0),
    'X100 (Cliver 1000-yr expon)':                 (100, 1000.0),
    'X180 (Largest-possible, Tschernitz)':         (180, 3000.0),
    'X285 (774 AD inferred)':                      (285, 10000.0),
}

print("Event                                         SXR class    Return period (yr)")
print("-" * 85)
for label, (sxr, T) in flare_tab.items():
    print(f"  {label:<43s}  X{sxr:<6d}  {T:>12.0f}")

# Visualization: return-period vs event size
sxrs = np.array([x[0] for x in flare_tab.values()])
Ts = np.array([x[1] for x in flare_tab.values()])
labels = list(flare_tab.keys())

fig, ax = plt.subplots(figsize=(11, 6))
ax.loglog(sxrs, Ts, 'bo-', ms=9, lw=1.5)
for sxr, T, label in zip(sxrs, Ts, labels):
    ax.annotate(label.split('(')[1].split(')')[0], (sxr, T),
                xytext=(5, 5), textcoords='offset points', fontsize=8)
ax.set_xlabel('GOES SXR class (X-number) / GOES SXR 등급')
ax.set_ylabel('Return period (yr) / 재현주기')
ax.set_title('Return period vs. SXR class / SXR 등급별 재현주기')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Miyake Δ¹⁴C Time Series / Miyake Δ¹⁴C 시계열

The Miyake events are signatures of extreme SEP events imprinted in tree-ring Δ¹⁴C. Fast rise (1-2 yr) followed by gradual (decades) decay via the carbon cycle. The confirmed events are:
- **774–775 AD**: Δ¹⁴C rise ≈ 12‰ (largest confirmed)
- **993–994 AD**: Δ¹⁴C rise ≈ 6‰
- **ca. 660 BC**: Δ¹⁴C rise ≈ 8‰
- **5259 BC, 7176 BC**: confirmed by Brehm, Paleari et al.

We model the Δ¹⁴C response as a fast production step followed by a slow exponential decay (carbon-cycle mixing).

Miyake 사건은 나무 나이테 Δ¹⁴C에 각인된 극한 SEP 사건의 signature. 빠른 상승(1-2년) 후 탄소 순환을 통한 점진적 감쇠(수십 년). 확인된 사건: 774–775 AD (Δ¹⁴C ≈ 12‰, 최대), 993–994 AD (~6‰), 660 BC (~8‰), 5259 BC, 7176 BC. Δ¹⁴C 응답을 빠른 생성 단계 뒤의 느린 지수 감쇠(탄소 순환 혼합)로 모델링.

In [ ]:
def delta14c_response(t, t_event, amplitude, tau_rise=1.0, tau_decay=30.0):
    """Model a Miyake-event Delta-14C response as fast rise and slow exponential decay.

    Args:
        t: Array of epoch (years BC/AD; negative = BC).
        t_event: Event epoch (year).
        amplitude: Peak Delta-14C increase (permille).
        tau_rise: Rise time (yr).
        tau_decay: Carbon-cycle mixing timescale (yr).

    Returns:
        Modeled Delta-14C(t) relative to the event baseline (permille).
    """
    dt = t - t_event
    response = np.zeros_like(dt, dtype=float)
    rise_mask = (dt >= 0) & (dt < tau_rise)
    decay_mask = dt >= tau_rise
    response[rise_mask] = amplitude * dt[rise_mask] / tau_rise
    response[decay_mask] = amplitude * np.exp(-(dt[decay_mask] - tau_rise) / tau_decay)
    return response


# Construct a 10,000-year synthetic Delta-14C record with the known Miyake events
years = np.arange(-8000, 2100, 1, dtype=float)
baseline = 15.0 * np.cos(2 * np.pi * (years - 1000) / 2300)  # grand-minima slow oscillation
noise = rng.normal(0, 2.0, size=years.shape)
delta14c = baseline + noise

# Superimpose confirmed Miyake events
events = {
    -7176:  7.0,  # Paleari/Brehm 2022
    -5259:  6.5,  # Brehm 2022
    -660:   8.0,  # O'Hare 2019
     774:  12.0,  # Miyake 2012 (largest)
     994:   6.0,  # Miyake 2013
}
for t_ev, amp in events.items():
    delta14c += delta14c_response(years, t_ev, amp)

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

axes[0].plot(years, delta14c, 'k-', lw=0.6, label='Synthetic $\\Delta^{14}$C with Miyake events')
for t_ev, amp in events.items():
    axes[0].axvline(t_ev, color='r', alpha=0.3)
    axes[0].annotate(f'{abs(t_ev)} {"BC" if t_ev<0 else "AD"} ({amp:.0f}‰)',
                     (t_ev, 50), fontsize=8, rotation=90, ha='right', va='top')
axes[0].set_xlabel('Year (BC/AD)')
axes[0].set_ylabel(r'$\Delta^{14}$C (\u2030)')
axes[0].set_title('Holocene $\\Delta^{14}$C with confirmed Miyake events / Holocene Δ¹⁴C 및 확인된 Miyake 사건')
axes[0].set_xlim(-8000, 2100)
axes[0].legend(loc='upper left')

# Zoom on the 774 AD event, showing the characteristic fast-rise/slow-decay
zoom_mask = (years >= 760) & (years <= 820)
axes[1].plot(years[zoom_mask], delta14c[zoom_mask], 'ko-', ms=3)
axes[1].axvline(775, color='r', ls='--', label='Event onset 774/775 AD')
axes[1].set_xlabel('Year AD')
axes[1].set_ylabel(r'$\Delta^{14}$C (\u2030)')
axes[1].set_title('Zoom on 774 AD Miyake event (cf. Fig. 3 of paper) / 774 AD Miyake 사건 확대')
axes[1].legend()
plt.tight_layout()
plt.show()

print("Confirmed Miyake events reproduced:")
for t_ev, amp in events.items():
    era = 'BC' if t_ev < 0 else 'AD'
    print(f"  {abs(t_ev):4d} {era}  Delta-14C peak: {amp:5.1f} permille")

## Part 5: Kepler Superflare Frequency on Sun-like Stars / Kepler 슈퍼플레어 빈도

Maehara et al. (2012) and subsequent *Kepler* studies found a unified flare energy distribution spanning nanoflares to superflares:

$$\frac{dN}{dE} \sim E^{-1.8}$$

Implied occurrence rates for Sun-like stars ($T_\text{eff}$ 5600–6000 K, $P_\text{rot}$ > 20 d):
- $10^{33}$ erg: once every ~2,000–6,000 years (Okamoto 2021 with gyrochronology)
- $10^{34}$ erg: once every ~500 years (Maehara 2015) to several thousand
- $10^{35}$ erg: once every few × 10⁴–10⁵ years

Maehara et al. (2012) 및 후속 Kepler 연구는 nanoflare부터 superflare까지 아우르는 통합된 플레어 에너지 분포 $dN/dE \sim E^{-1.8}$을 발견. Sun-like 별에 대한 암시된 발생률: $10^{33}$ erg는 약 2000-6000년에 한 번, $10^{34}$ erg는 약 500년에 한 번, $10^{35}$ erg는 수만-수십만 년에 한 번.

In [ ]:
def superflare_freq(E, N_nano_per_day=100, E_nano=1e24, alpha=1.8):
    """Extrapolate the universal dN/dE ~ E^-1.8 from nanoflares to superflares.

    Args:
        E: Flare energy (erg).
        N_nano_per_day: Number of nanoflares per day from a solar active region.
        E_nano: Characteristic nanoflare energy (erg).
        alpha: Power-law index.

    Returns:
        Occurrence rate (events per year per star) at or above E.
    """
    N_nano_per_year = N_nano_per_day * 365
    # Cumulative N(>E) for power-law: N(>E) = N(>E_nano) * (E/E_nano)^(-(alpha-1))
    return N_nano_per_year * (E / E_nano)**(-(alpha - 1))


E_range = np.logspace(24, 36, 300)
rate = superflare_freq(E_range)
T_return = 1.0 / rate

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.loglog(E_range, T_return, 'b-', lw=2)
ax.axvline(1e24, color='gray', ls=':', alpha=0.5)
ax.axvline(1e27, color='gray', ls=':', alpha=0.5)
ax.axvline(1e29, color='gray', ls=':', alpha=0.5)

# Key markers (from Table 10 and §3.2.5)
markers = [
    (4.3e32, 1, '4 Nov 2003 (X35)'),
    (5e32, 100, 'Carrington 1859'),
    (1.9e33, 1e4, '774 AD inferred'),
    (1e34, 500, 'Superflare threshold (Maehara 2015)'),
    (1e35, 1e5, r'Largest Kepler superflare ($\sim 10^{35}$ erg)'),
]
for E_ev, T_ev, label in markers:
    ax.plot(E_ev, T_ev, 'ro', ms=10, mec='k')
    ax.annotate(label, (E_ev, T_ev), xytext=(8, -3),
                textcoords='offset points', fontsize=9)

ax.set_xlabel('Flare energy $E$ (erg)')
ax.set_ylabel('Return period (years)')
ax.set_title('Unified solar + Sun-like star flare return period (from $dN/dE \\propto E^{-1.8}$)\n'
             '태양 + Sun-like 별 플레어의 통합된 return period')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print("Return period vs. flare bolometric energy (from extrapolation):")
for E_query in [4e32, 5e32, 2e33, 1e34, 1e35]:
    T = 1.0 / superflare_freq(E_query)
    print(f"  E = {E_query:.1e} erg  →  T = {T:.2e} yr")

## Part 6: Societal Impact Estimation / 사회적 영향 추정

NRC (2008) and Lloyd's (2010) estimated the first-year economic loss from a Carrington-class event at **$1–2 trillion** with recovery 4–10 years. Here we construct a schematic loss function combining: (a) Dst-driven GIC impact (power grid collapse probability rises non-linearly with |Dst|), (b) SEP-driven spacecraft and aircrew impact (linear in F₂₀₀), (c) radio burst impact on GPS (linear above 10⁶ sfu).

This is a pedagogical illustration — real impact estimation requires detailed infrastructure modeling.

NRC (2008) 및 Lloyd's (2010)는 Carrington급 사건의 첫 해 경제 손실을 4-10년 복구 기간을 포함하여 **1-2조 달러**로 추정. 여기서는 (a) Dst 기반 GIC 영향 (|Dst|에 따라 비선형 증가하는 전력망 붕괴 확률), (b) SEP 기반 우주선/승무원 영향 (F₂₀₀에 선형), (c) 전파 버스트의 GPS 영향 (10⁶ sfu 이상에서 선형)을 결합한 개념적 손실 함수 구성. 실제 영향 추정은 상세한 인프라 모델링 필요 — 교육적 예시.

In [ ]:
def loss_function(Dst_nT, F200_cm2, radio_sfu, carrington_loss_usd=1.5e12):
    """Schematic economic-loss function combining storm, SEP, radio drivers.

    Args:
        Dst_nT: Minimum Dst (nT, negative).
        F200_cm2: > 200 MeV proton fluence (cm^-2).
        radio_sfu: Peak ~1 GHz radio burst flux density (sfu).
        carrington_loss_usd: Reference loss for Carrington (Lloyd's/NRC).

    Returns:
        Estimated economic loss in USD.
    """
    # GIC power-grid damage: non-linear ramp above -250 nT
    dst_factor = max(0.0, (abs(Dst_nT) - 250.0) / 700.0) ** 1.8
    # SEP component: linear in fluence, normalized to 1956 GLE
    sep_factor = F200_cm2 / 1.4e8
    # Radio component: linear above 10^6 sfu
    radio_factor = max(0.0, (radio_sfu - 1e6) / 1e6)
    total = dst_factor + 0.2 * sep_factor + 0.1 * radio_factor
    return total * carrington_loss_usd


scenarios = {
    'March 1989 (Quebec blackout)':  (-589,  1e7, 1e4),
    'Carrington 1859':               (-949,  2e9, 1e5),
    '23 July 2012 (if Earth dir.)':  (-1200, 5e8, 1e5),
    'August 1972 (worst obs.)':      (-1400, 1.4e8, 1e5),
    '774 AD SEP (cosmogenic)':       (-950,  1e10, 1e6),
    'Theoretical maximum':           (-2500, 1e11, 1e7),
}

print("Scenario                                Dst (nT)   F200 (cm^-2)   Radio (sfu)   Loss (USD)")
print("-" * 105)
results = []
for name, (dst, f200, radio) in scenarios.items():
    loss = loss_function(dst, f200, radio)
    results.append((name, dst, f200, radio, loss))
    print(f"  {name:<39s}  {dst:>7d}   {f200:>11.1e}   {radio:>11.1e}   ${loss:>11.2e}")

# Plot
names = [r[0] for r in results]
losses = np.array([r[4] for r in results])
dsts = np.array([abs(r[1]) for r in results])

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(names, losses / 1e12, color='darkred', alpha=0.7)
for i, loss in enumerate(losses):
    ax.text(loss / 1e12 * 1.05, i, f'${loss/1e12:.2f}T', va='center', fontsize=9)
ax.set_xlabel('Estimated economic loss (trillion USD)')
ax.set_xscale('log')
ax.set_title('Schematic infrastructure loss vs. event magnitude\n인프라 손실 대 사건 규모 (개념적)')
ax.set_xlim(left=1e-2)
plt.tight_layout()
plt.show()

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Implementation / 현대 구현 |
|---|---|---|
| Flare energy distribution / 플레어 에너지 분포 | $dN/dE \propto E^{-1.8}$, Gopalswamy modified exponential | Power-law + rolloff fit; 100-yr X44, 1000-yr X100 |
| Extreme value analysis / 극값 분석 | GEV/Gumbel for storm Dst (Thomson 2011) | `scipy.stats.gumbel_r`, MLE fit, return-level curve |
| Carrington return period / Carrington 재현주기 | X45 ≈ 100-yr event; Dst = −949 nT ≈ 500-yr (expon) or 80-yr (PL) | OFD intersection with $Y = -2$ or $-3$ |
| Miyake events / Miyake 사건 | 774/993/660 BC/5259 BC/7176 BC confirmed; $\Delta^{14}$C ≈ 6–12‰ | Fast-rise/slow-decay response model with carbon-cycle $\tau \sim 30$ yr |
| Superflare frequency / 슈퍼플레어 빈도 | Kepler: $10^{33}$ erg every ~6000 yr (Okamoto) | Extrapolate $dN/dE \propto E^{-1.8}$ from nanoflares |
| Infrastructure loss / 인프라 손실 | $1–2T$ for Carrington-class (NRC 2008, Lloyd's 2010) | Combined Dst + SEP + radio loss function; 774 AD = ~100× Carrington |

**Key quantitative reproduction / 핵심 정량적 재현**:
- Flare OFD 100-yr, 1000-yr, 10,000-yr events: X44, X100, X200 (Cliver et al. Table 10, fit-dependent).
- 774 AD SEP event inferred from F₂₀₀–SXR correlation: X285 ± 140, $1.9 \pm 0.7 \times 10^{33}$ erg.
- Miyake events reproduced at correct epochs with appropriate amplitudes (12‰ for 774 AD).
- 1000-yr superflare ($\sim 10^{33}$ erg) frequency agrees with Maehara 2015 / Notsu 2019 Kepler result.
- Economic loss scaling: 774 AD SEP (if Earth-directed) implies ~100× Carrington-class infrastructure damage.